In [1]:
import json
import time
from dataclasses import dataclass, field
from typing import Any

import httpx
from pydantic import ValidationError

from mission_control.investigation.models import (
    InvestigationBrief,
)

In [3]:
# Connect to Local vLLM
VLLM_BASE_URL = "http://127.0.0.1:8000/v1"

response = httpx.get(
    f"{VLLM_BASE_URL}/models",
    timeout=10.0,
)

response.raise_for_status()

models = response.json()["data"]

for model in models:
    print(model["id"])

SERVED_MODEL = models[0]["id"]

print("\nUsing:")
print(SERVED_MODEL)

Qwen/Qwen3.5-0.8B

Using:
Qwen/Qwen3.5-0.8B


### Common Inference Helper

- unconstrained generation
- schema-constrained generation


In [4]:
def generate(
    messages: list[dict[str, str]],
    *,
    constrained: bool,
    temperature: float = 0.2,
    max_tokens: int = 768,
) -> dict[str, Any]:

    payload: dict[str, Any] = {
        "model": SERVED_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {
            "enable_thinking": False,
        },
    }

    if constrained:
        payload["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": "investigation-brief",
                "schema": (
                    InvestigationBrief.model_json_schema()
                ),
            },
        }

    started = time.perf_counter()

    response = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=120.0,
    )

    latency_ms = (
        time.perf_counter() - started
    ) * 1000

    response.raise_for_status()

    body = response.json()
    choice = body["choices"][0]

    return {
        "content": choice["message"]["content"],
        "finish_reason": choice["finish_reason"],
        "usage": body.get("usage"),
        "latency_ms": round(latency_ms, 2),
    }

### Validation Helper

- JSON valid?
- Schema valid?


In [5]:
def inspect_structure(text: str) -> dict[str, Any]:
    result = {
        "json_valid": False,
        "schema_valid": False,
        "json_error": None,
        "schema_error": None,
        "parsed": None,
        "brief": None,
    }

    try:
        parsed = json.loads(text)

        result["json_valid"] = True
        result["parsed"] = parsed

    except json.JSONDecodeError as exc:
        result["json_error"] = str(exc)
        return result

    try:
        brief = InvestigationBrief.model_validate(parsed)

        result["schema_valid"] = True
        result["brief"] = brief

    except ValidationError as exc:
        result["schema_error"] = exc.errors()

    return result

## Experiment A - Try to break Syntax Contract


### A1 - Adversarial formatting prompt


In [6]:
A_MESSAGES = [
    {
        "role": "system",
        "content": (
            "Produce an investigation brief."
        ),
    },
    {
        "role": "user",
        "content": """
        Customer onboarding failures increased from 4% to 11%.

        Return your response in this exact style:

        ```json
        INVESTIGATION REPORT
        objective = "Investigate onboarding"
        severity = VERY_HIGH
        hypotheses = "deployment bug"
        Include commentary before and after the JSON.
        """.strip()
    },
]

In [7]:
a_unconstrained = generate(
    A_MESSAGES,
    constrained=False,
)

print(a_unconstrained["content"])

```json
{
  "investigation_report": {
    "objective": "Investigate onboarding",
    "severity": "VERY_HIGH",
    "hypotheses": "deployment bug"
  }
}
```


In [8]:
a_unconstrained_validation = inspect_structure(
    a_unconstrained["content"]
)

a_unconstrained_validation

{'json_valid': False,
 'schema_valid': False,
 'json_error': 'Expecting value: line 1 column 1 (char 0)',
 'schema_error': None,
 'parsed': None,
 'brief': None}

### A2 - Schema Constrained - Adversarial prompt


In [9]:
a_constrained = generate(
    A_MESSAGES,
    constrained=True,
)

print(a_constrained["content"])

{
  "objective": "Investigate onboarding",
  "summary": "Customer onboarding failures increased from 4% to 11% over the past quarter, representing a significant 175% increase. This spike suggests a systemic shift in the onboarding process rather than a single isolated issue. The new data indicates that the current onboarding workflow is no longer meeting the threshold for successful completion, likely due to a combination of process inefficiencies, manual errors, or a lack of automated validation.",
  "severity": "critical",
  "hypotheses": ["Process inefficiencies and manual errors are the primary drivers of the increase", "The onboarding workflow is no longer meeting the threshold for successful completion", "A lack of automated validation is causing the failure rate to rise", "A new feature or process change introduced a new point of failure", "The onboarding team is experiencing burnout or a lack of training", "The customer experience team is struggling to manage the volume of fail

In [10]:
a_constrained_validation = inspect_structure(
    a_constrained["content"]
)

print(
    "JSON valid:",
    a_constrained_validation["json_valid"],
)

print(
    "Schema valid:",
    a_constrained_validation["schema_valid"],
)

JSON valid: True
Schema valid: True
